<a href="https://colab.research.google.com/github/Marcin19721205/WSBNeuronowe/blob/main/04_R%C3%B3wnowa%C5%BCenie_klas_caly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wprowadzenie

Kolejnym ważnym aspektem niemal każdego eksperymentu uczenia maszynowego jest możliwość równoważenia klas w przypadku, gdy ich proporcje są zaburzone. W większości rzeczywistych zadań, część klas może występować w niewielkiej ilości. Niestety, najczęściej będą to te najbardziej wartościowe i istotne. Istnieją narzędzia, które pozwalają radzić sobie z takimi zjawiskami.

W tym notebooku zapoznamy się z podstawowymi technikami, które mogą nam pomóc przywrócić właściwe proporcje w danych.

In [1]:
%pip install imblearn

In [2]:
%pip install tensorflow_addons

ERROR: Could not find a version that satisfies the requirement tensorflow_addons (from versions: none)
ERROR: No matching distribution found for tensorflow_addons


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
#import tensorflow_addons as tfa
import gc
import tensorflow.keras as krs

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, accuracy_score, precision_score, recall_score

In [5]:
%matplotlib inline

# Wczytanie danych



W ramach tego ćwiczenia będziemy pracować na zbiorze danych do klasyfikacji wieloklasowej, gdzie porporcje pomiędzy klasami są dość silnie zaburzone.

<div class='alert alert-block alert-warning'>
    Wczytaj zbiór danych o nazwie <b>imbalanced_dataset.csv</b>. Sprawdź proporcje pomiędzy klasami, zawartymi w kolumnie <b>y</b>.
</div>

Dane są dostępne pod adresem `https://drive.google.com/uc?id=1u2M_HRvD_MXJ7Kztbyr7i9FvvW0Ms6TC&export=download`

In [7]:
data = pd.read_csv("https://drive.google.com/uc?id=1u2M_HRvD_MXJ7Kztbyr7i9FvvW0Ms6TC&export=download")

In [8]:
data["y"].value_counts(normalize=True).sort_index().round(4)  # wyświetl proporcje (liczność względna) klas

,proportion
y,
0,0.6980
1,0.2012
2,0.1008


<div class='alert alert-block alert-warning'>
    <b>Zadanie</b>:
    <ol>
        <li>oddziel kolumnę y od całej reszty danych. Zapisz ją pod zmienną y</li>
        <li>dane, które pozostają - zapisz pod zmienną X</li>
        <li>zrób rzutowanie typów zmiennej X na typ <code>np.float32</code> ze względu na kompatybilność z tensorflow</li>
        <li>zamień klasy wektora <code>y</code> na postać one-hot i zapisz ponownie pod zmienną y</li>
    </ol>
</div>

In [16]:
y = data["y"]  # wydziel y
X = data.drop(columns=["y"]).astype(np.float32)  # wydziel X + float32 (DataFrame pod assert)



In [17]:
y = tf.keras.utils.to_categorical(y.to_numpy().astype(int), num_classes=3)  # one-hot y


Sprawdzenie poprawności wyników:

In [26]:
assert X.shape == (5000, 20)
assert (X.dtypes == np.float32).all()

assert y.shape == (5000, 3)

<div class='alert alert-block alert-warning'>
    Podziel dane na train i test w proporcji <code>train = 0.8% zbioru, random_state = 123</code>
</div>

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=123)  # podziel zbiór


Sprawdzenie poprawności wyniku

In [22]:
assert X_train.shape == (4000, 20)
assert y_train.shape == (4000, 3)

assert X_test.shape == (1000, 20)
assert y_test.shape == (1000, 3)

assert X_train.values.dtype == np.float32
assert X_test.values.dtype == np.float32

# Bazowe modele

Zaczniemy od zbudowania bazowego modelu, który będzie operował na oryginalnych danych, bez równoważenia. Stwrzomy dwa modele:

1. Zawsze przewidujący najczęstszą klasę
2. Prostą sieć neuronową do klasyfikacji wieloklasowej

Sieć neuronową o zadanej architekturze będziemy szkolić od zera kilkukrotnie, odpowiednio manipulując wcześniej danymi.

<div class='alert alert-block alert-danger'>
    <b>UWAGA</b> w tym ćwiczeniu nie skupiamy się na stworzeniu jak najlepszej architektury sieci neurnowej dla zadanego problemu. Chcemy za to zbadać wpływ równowagi klas lub jej braku na jakość predykcji. Nie skupiaj się więc na aspekcie doboru jak najlepszej sieci, ale na operacjach na danych, które za chwilę będziemy wykonywać.
</div>

## Prosty klasyfikator

<div class='alert alert-block alert-warning'>
    Zbuduj klasyfikator, zawsze przewidujący najczęściej występującą klasę. Wykorzystaj implementajcę <code>sklearn.dummy.DummyClassifier</code>
</div>


In [24]:
from sklearn.dummy import DummyClassifier  # dummy

dummy = DummyClassifier(strategy="most_frequent")  # zawsze najczęstsza klasa
dummy.fit(X_train, y_train.argmax(1) if getattr(y_train, "ndim", 1) > 1 else y_train)  # fit (obsługa one-hot)



DummyClassifier(strategy='most_frequent')

Sprawdzenie poprawności wyniku

In [27]:
yhat_dummy = dummy.predict(X_test)
assert np.round(accuracy_score(y_test.argmax(axis=1), yhat_dummy), 3) == 0.694
print(classification_report(y_test.argmax(axis=1), yhat_dummy, zero_division=0))

clf_rep = classification_report(y_test.argmax(axis=1), yhat_dummy, zero_division=0, output_dict=True)
assert np.round(clf_rep['0']['precision'], 3) == 0.694
assert clf_rep['1']['precision'] == 0.0

assert clf_rep['0']['recall'] == 1.0
assert clf_rep['1']['recall'] == 0.0

              precision    recall  f1-score   support

           0       0.69      1.00      0.82       694
           1       0.00      0.00      0.00       215
           2       0.00      0.00      0.00        91

    accuracy                           0.69      1000
   macro avg       0.23      0.33      0.27      1000
weighted avg       0.48      0.69      0.57      1000



<div class='alert alert-block alert-info'>
    W powyższych wynikach widać trzy niepokojące rzeczy:
    
<ol>
<li>Głupi klasyfikator potrafi "osiągnąć" trafnośc na poziomie 69%</li>
<li>Metryki takie jak trafność są bezużyteczne w przypadku braku zrównoważenia klas</li>
<li>Dopiero łączne wykorzystanie metryk precyzji, czułości oraz F1 pozwala zobaczyć skalę problemu</li>
</ol>
</div>

## Sieć neuronowa

<div class='alert alert-block alert-warning'>
    Przygotuj funkję, która buduje i zwraca gotową sieć neuronową o następującej specyfikacji:

<ol>
<li>Warstwy: <code>BatchNorm - Dense(32, relu) - Dense(16, relu) - Dense(3, softmax)</code></li>
<li>Dodatkowe opcje: <code>optymalizator=Adam, koszt=categorical_crossentropy, metryki: accuracy, F1Score(num_classes=3, average=macro)</code></li>
</ol>
</div>

<div class='alert alert-block alert-info'>
    Metryka F1Score zawarta są w bardzo przydatnym pakiecie <code>tensorflow_addons</code>. Warto przeczytać dokumentację tego narzędzia.
</div>


In [92]:
import tensorflow.keras as krs

def build_model():
    reg = tf.keras.regularizers.l2(1e-5)  # L2
    model = tf.keras.Sequential([  # model
        tf.keras.layers.BatchNormalization(input_shape=(X_train.shape[1],)),  # BatchNorm
        tf.keras.layers.Dense(32, activation="relu", kernel_regularizer=reg),  # Dense 32 relu + L2
        tf.keras.layers.Dense(16, activation="relu", kernel_regularizer=reg),  # Dense 16 relu + L2
        tf.keras.layers.Dense(3, activation="softmax", kernel_regularizer=reg),  # Dense 3 softmax + L2
    ])  # layers
    model.compile(  # compile
        optimizer=tf.keras.optimizers.Adam(),  # Adam
        loss="categorical_crossentropy",  # cce
        #metrics=['accuracy', krs.metrics.F1Score(num_classes=3, average='macro', threshold=0.5)] # Added metrics
    )  # compile
    return model  # return

In [93]:
model1 = build_model()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/normalization/batch_normalization.py:142: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Sprawdzenie poprawności wyniku:

In [94]:
assert 'batch_normalization' in model1.layers[0].name

assert 'dense' in model1.layers[1].name
assert model1.layers[1].units == 32
assert model1.layers[1].activation is tf.keras.activations.relu

assert 'dense' in model1.layers[2].name
assert model1.layers[2].units == 16
assert model1.layers[2].activation is tf.keras.activations.relu

assert 'dense' in model1.layers[3].name
assert model1.layers[3].units == 3
assert model1.layers[3].activation is tf.keras.activations.softmax

<div class='alert alert-block alert-warning'>
    Wyszkol przygotowany model przez 5 epok (batch size 32) na zbiorze treningowym. Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m1_eval.
</div>

In [95]:
model1.fit(X_train, y_train, epochs=5, batch_size=32, verbose=1)  # przeszkol model
m1_eval = model1.evaluate(X_test, y_test, return_dict=True, verbose=1)  # zapisz wartości metryk

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 1.0652
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.7075
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6349
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.5782
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.5451
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.5341  


In [96]:
m1_eval

{'loss': 0.5364526510238647}

Sprawdzenie poprawności wyniku:

In [98]:
assert "loss" in m1_eval  # jest wynik
assert m1_eval["loss"] <= 1.0  # sensowny próg dla loss


<div class='alert alert-block alert-warning'>
    Dokonaj predykcji na zbiorze testowym. Wszystkie obiekty, które osiągną próg pewności >=0.5 zalicz do klasy 1. Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.
</div>

In [99]:
y_proba = model1.predict(X_test, verbose=0)  # predykcja prawdopodobieństw
y_pred = np.where(y_proba.max(1) >= 0.5, 1, y_proba.argmax(1))  # jeśli pewność>=0.5 -> klasa 1
print(classification_report(y_test.argmax(1), y_pred, zero_division=0))  # raport


              precision    recall  f1-score   support

           0       0.46      0.02      0.04       694
           1       0.21      0.93      0.34       215
           2       0.30      0.07      0.11        91

    accuracy                           0.22      1000
   macro avg       0.32      0.34      0.16      1000
weighted avg       0.39      0.22      0.11      1000



<div class='alert alert-block alert-info'>
    Jak widać, czułośc (ang. *recall*) i precyzja  (ang. *precision*) dla klas o małej liczności nie są zbyt dobre. Spróbujemy je poprawić <b>równoważąc klasy w próbce uczącej.</b>
</div>

# Równoważenie klas

Poniżej zostaną zaprezentowane sposoby równoważenia klas, należące do kategorii opisywanych na wykładzie. Zastosujemy kilka z nich i sprawdzimy, czy dają oczekiwane rezultaty.

Zaczniemy od zaimportowania biblioteki, w której zawarte są odpowiednie narzędzia.

In [100]:
import imblearn

## Oversampling

Pierwszą z metod będzie 'dolosowywanie' obiektów z klasy mniejszościowej. Zrobimy to dwoma sposobami.

### Random

Pierwszy sposób dolosowtwania klasy mniejszościowej polega na losowym wyborze obiektów z klas mniejszościowych tak długo, aż poszczególne ilości się zrównoważą.

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Z modułu <code>imblearn.over_sampling</code> zaimportuj obiekt <code>RandomOverSampler</code>. </li>
        <li>Utwórz obiekt klasy <code>RandomOverSampler</code> ustawiając random_state=999</li>
        <li>Wywołaj funkcję <code>.fit_transform(..., ...)</code> obiektu RandomOverSamplera, szkoląc go na treningowych danych X i y</li>
        <li>W procesie szkolenia oversampler przeliczy klasy i dokona ich równoważenia, zwracając nowy zbiór danych</li>
        <li>Zapisz docelowy zbiór danych pod zmiennymi <code>X_train_ros, y_train_ros</code></li>
        <li>Sprawdź proporcje klas w zbiorze treningowym - zapisz je w postaci <b>słowika (dict) pod zmienną ycnt_ros: [klucz: numer klasy]: [wartość: [%] w zbiorze treningowym]</b></li>
    </ol>
</div>

In [102]:
from imblearn.over_sampling import RandomOverSampler  # import ROS

ros = RandomOverSampler(random_state=999)  # ROS
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train.argmax(1))  # oversample train


In [103]:
cnt = np.bincount(y_train_ros, minlength=3)  # counts
ycnt_ros = {i: cnt[i] / cnt.sum() for i in range(3)}  # proportions


Sprawdzenie poprawności wyniku:

In [104]:
for i in range(3):
    assert round(ycnt_ros[i], 3) == 0.333

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Wykorzystując napisaną wcześniej funkcję- utwórz nowy obiekt sieci neurnowej</li>
        <li>Wyszkol przygotowany model przez 5 epok (batch size 32) na zrównoważonym zbiorze treningowym. </li>
        <li>Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m2_eval</li>
        <li>Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.</li>
    </ol>
</div>

In [105]:
model2 = build_model()  # nowy model
model2.fit(X_train_ros, tf.keras.utils.to_categorical(y_train_ros, num_classes=3), epochs=5, batch_size=32, verbose=1)  # wytrenuj model
m2_eval = model2.evaluate(X_test, y_test, return_dict=True, verbose=0)  # zapisz wartości metryk
print(classification_report(y_test.argmax(1), model2.predict(X_test, verbose=0).argmax(1), zero_division=0))  # raport


/usr/local/lib/python3.12/dist-packages/keras/src/layers/normalization/batch_normalization.py:142: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 1.0776
Epoch 2/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.7887
Epoch 3/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.6536
Epoch 4/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5947
Epoch 5/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5469
              precision    recall  f1-score   support

           0       0.93      0.81      0.87       694
           1       0.73      0.72      0.72       215
           2       0.40      0.81      0.54        91

    accuracy                           0.79      1000
   macro avg       0.69      0.78      0.71      1000
weighted avg       0.84      0.79      0.81      1000



In [106]:
m2_eval

{'loss': 0.5572971701622009}

In [108]:
print(classification_report(y_test.argmax(1), model2.predict(X_test, verbose=0).argmax(1), zero_division=0))  # raport


              precision    recall  f1-score   support

           0       0.93      0.81      0.87       694
           1       0.73      0.72      0.72       215
           2       0.40      0.81      0.54        91

    accuracy                           0.79      1000
   macro avg       0.69      0.78      0.71      1000
weighted avg       0.84      0.79      0.81      1000



<div class='alert alert-block alert-info'>
    Wyniki powinny się zmienić w stosunku do scenariusza bazowego - najprawdopodobniej spadła dokładność (ang. <i>accuracy</i>) ale wzrosła czułość i precyzja dla co najmniej jednej klasy mniejszościowej (1 i 2). To jest spodziewany efekt. Będziemy szukać dalej, czy inne procedury równoważenia zapewnią lepsze wyniki.
</div>

### SMOTE

Durgą procedurą równoważenia próbek, którą wykorzystamy będzie SMOTE, omawiane na wykładach. Ta metoda tworzy syntetyczne próbki, powtałe na przecięciu odcinków łączących obiekty z klasy mniejszościowej.

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Z modułu <code>imblearn.over_sampling</code> zaimportuj obiekt <code>SMOTE</code>. </li>
        <li>Utwórz obiekt klasy <code>SMOTE</code> ustawiając random_state=999</li>
        <li>Wywołaj funkcję <code>.fit_transform(..., ...)</code> obiektu SMOTE, szkoląc go na treningowych danych X i y</li>
        <li>W procesie szkolenia oversampler przeliczy klasy i dokona ich równoważenia, zwracając nowy zbiór danych</li>
        <li>Zapisz docelowy zbiór danych pod zmiennymi <code>X_train_smote, y_train_smote</code></li>
        <li>Sprawdź proporcje klas w zbiorze treningowym - zapisz je w postaci <b>słowika (dict) pod zmienną ycnt_smote: [klucz: numer klasy]: [wartość: [%] w zbiorze treningowym]</b></li>
    </ol>
</div>

In [109]:
from imblearn.over_sampling import SMOTE  # import SMOTE

smote = SMOTE(random_state=999)  # SMOTE
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train.argmax(1))  # SMOTE train


In [110]:
cnt = np.bincount(y_train_smote, minlength=3)  # counts
ycnt_smote = {i: cnt[i] / cnt.sum() for i in range(3)}  # proportions


Sprawdzenie poprawności wyniku

In [111]:
for i in range(3):
    assert round(ycnt_smote[i], 3) == 0.333

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Wykorzystując napisaną wcześniej funkcję- utwórz nowy obiekt sieci neurnowej</li>
        <li>Wyszkol przygotowany model przez 5 epok (batch size 32) na zrównoważonym zbiorze treningowym. </li>
        <li>Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m3_eval</li>
        <li>Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.</li>
    </ol>
</div>

In [112]:


model3 = build_model()  # nowy model
model3.fit(X_train_smote, tf.keras.utils.to_categorical(y_train_smote, num_classes=3), epochs=5, batch_size=32, verbose=1)  # wyszkol model
m3_eval = model3.evaluate(X_test, y_test, return_dict=True, verbose=0)  # zapisz metryki
print(classification_report(y_test.argmax(1), model3.predict(X_test, verbose=0).argmax(1), zero_division=0))  # raport


Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/normalization/batch_normalization.py:142: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


263/263 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 1.0631
Epoch 2/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.7585
Epoch 3/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6075
Epoch 4/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5251
Epoch 5/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4804
              precision    recall  f1-score   support

           0       0.92      0.81      0.86       694
           1       0.72      0.71      0.71       215
           2       0.43      0.80      0.56        91

    accuracy                           0.79      1000
   macro avg       0.69      0.78      0.71      1000
weighted avg       0.83      0.79      0.80      1000



In [113]:
m3_eval

{'loss': 0.5023283958435059}

In [115]:
print(classification_report(y_test.argmax(1), model3.predict(X_test, verbose=0).argmax(1), zero_division=0))  # raport


              precision    recall  f1-score   support

           0       0.92      0.81      0.86       694
           1       0.72      0.71      0.71       215
           2       0.43      0.80      0.56        91

    accuracy                           0.79      1000
   macro avg       0.69      0.78      0.71      1000
weighted avg       0.83      0.79      0.80      1000



<div class='alert alert-block alert-info'>
    Wyniki powinny się zmienić w stosunku do scenariusza bazowego - najprawdopodobniej spadła dokładność (ang. <i>accuracy</i>) ale wzrosła czułość i precyzja dla co najmniej jednej klasy mniejszościowej (1 i 2). To jest spodziewany efekt. Będziemy szukać dalej, czy inne procedury równoważenia zapewnią lepsze wyniki.<br>
    Porównaj otrzymane wyniki z RandomOverSampling'iem. Czy jest lepiej, czy gorzej? Jeśli tak, to w jakim zakresie (w odniesieniu do której klasy?).
</div>

## Under sampling

Kolejna grupa procedur to zmniejszenie liczności klasy większościowej - odrzucenie nadmiarowych obserwaci, aby zredukować ją do takiej samej liczności, jak klasa mniejszościowa. Ta metoda niestety powoduje odrzucenie znacznej ilości użytecznych danych - z tego powodu może być czasem problematyczna.

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Z modułu <code>imblearn.over_sampling</code> zaimportuj obiekt <code>RandomUnderSampler</code>. </li>
        <li>Utwórz obiekt klasy <code>RandomUnderSampler</code> ustawiając random_state=999</li>
        <li>Wywołaj funkcję <code>.fit_transform(..., ...)</code> obiektu RandomUnderSampler, szkoląc go na treningowych danych X i y</li>
        <li>W procesie szkolenia oversampler przeliczy klasy i dokona ich równoważenia, zwracając nowy zbiór danych</li>
        <li>Zapisz docelowy zbiór danych pod zmiennymi <code>X_train_rus, y_train_rus</code></li>
        <li>Sprawdź proporcje klas w zbiorze treningowym - zapisz je w postaci <b>słowika (dict) pod zmienną ycnt_rus: [klucz: numer klasy]: [wartość: [%] w zbiorze treningowym]</b></li>
    </ol>
</div>

In [116]:
from imblearn.under_sampling import RandomUnderSampler  # import RUS

rus = RandomUnderSampler(random_state=999)  # RUS
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train.argmax(1))  # undersample train


In [117]:
cnt = np.bincount(y_train_rus, minlength=3)  # counts
ycnt_rus = {i: cnt[i] / cnt.sum() for i in range(3)}  # proportions


Sprawdzenie poprawności wyniku:

In [118]:
for i in range(3):
    assert round(ycnt_rus[i], 3) == 0.333

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Wykorzystując napisaną wcześniej funkcję- utwórz nowy obiekt sieci neurnowej</li>
        <li>Wyszkol przygotowany model przez 5 epok (batch size 32) na zrównoważonym zbiorze treningowym. </li>
        <li>Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m4_eval</li>
        <li>Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.</li>
    </ol>
</div>

In [119]:
model4 = build_model()  # nowy model
model4.fit(X_train_rus, tf.keras.utils.to_categorical(y_train_rus, num_classes=3), epochs=5, batch_size=32, verbose=1)  # wyszkol model
m4_eval = model4.evaluate(X_test, y_test, return_dict=True, verbose=0)  # zapisz metryki


/usr/local/lib/python3.12/dist-packages/keras/src/layers/normalization/batch_normalization.py:142: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
39/39 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 1.1912
Epoch 2/5
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.0847
Epoch 3/5
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.0375
Epoch 4/5
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9892
Epoch 5/5
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9347


In [120]:
m4_eval

{'loss': 0.9216205477714539}

In [122]:
print(classification_report(y_test.argmax(1), model4.predict(X_test, verbose=0).argmax(1), zero_division=0))  # raport


              precision    recall  f1-score   support

           0       0.85      0.54      0.66       694
           1       0.41      0.58      0.48       215
           2       0.19      0.56      0.29        91

    accuracy                           0.55      1000
   macro avg       0.49      0.56      0.48      1000
weighted avg       0.70      0.55      0.59      1000



<div class='alert alert-block alert-info'>
   W tym przypadku wyniki powinny być znacznie gorsze, niż przy wykorzystaniu wcześniejszych podejść oraz w modelu bazowym. Wynika to z faktu, że RandomUnderSampler odrzuca obiekty (rekordy), które mogą nieść ze sobą bardzo użyteczne informacje.
    <br>
    <br>
    Nie znaczy to, że UnderSampling nie jest przydatny - najcześciej wykorzystuje się go w sytuacjach, gdy mamy bardzo dużo danych, które niekoniecznie muszą być użyteczne (np. macierze rzadkie w systemach rekomendacyjnych, etc.).
    <br>
    <br>
    W tym konkretnym przypadku - raczej nam się nie przyda.
</div>

## SMOTETomek - upsampling i undersampling jedncześnie

Jak łatwo się domyślić, opisane wyżej metody można połączyć, stosując jednocześnie syntetyczny oversampling oraz redukcję niektórych obserwacji z klasy większościowej. Spróbujmy i sprawdźmy, czy ta metoda da lepsze rezultaty niż np. wyłącznie SMOTE.

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Z modułu <code>imblearn.over_sampling</code> zaimportuj obiekt <code>SMOTETomek</code>. </li>
        <li>Utwórz obiekt klasy <code>SMOTETomek</code> ustawiając random_state=999</li>
        <li>Wywołaj funkcję <code>.fit_transform(..., ...)</code> obiektu SMOTETomek, szkoląc go na treningowych danych X i y</li>
        <li>W procesie szkolenia oversampler przeliczy klasy i dokona ich równoważenia, zwracając nowy zbiór danych</li>
        <li>Zapisz docelowy zbiór danych pod zmiennymi <code>X_train_smotet, y_train_smotet</code></li>
        <li>Sprawdź proporcje klas w zbiorze treningowym - zapisz je w postaci <b>słowika (dict) pod zmienną ycnt_smotet: [klucz: numer klasy]: [wartość: [%] w zbiorze treningowym]</b></li>
    </ol>
</div>

In [123]:
from imblearn.combine import SMOTETomek  # import SMOTETomek

smotet = SMOTETomek(random_state=999)  # SMOTETomek
X_train_smotet, y_train_smotet = smotet.fit_resample(X_train, y_train.argmax(1))  # resample train


In [124]:
cnt = np.bincount(y_train_smotet, minlength=3)  # counts
ycnt_smotet = {i: cnt[i] / cnt.sum() for i in range(3)}  # proportions


Sprawdzenie poprawności wyniku:

In [125]:
for i in range(3):
    assert round(ycnt_smotet[i], 3) == 0.333

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Wykorzystując napisaną wcześniej funkcję- utwórz nowy obiekt sieci neurnowej</li>
        <li>Wyszkol przygotowany model przez 5 epok (batch size 32) na zrównoważonym zbiorze treningowym. </li>
        <li>Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m5_eval</li>
        <li>Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.</li>
    </ol>
</div>

In [126]:
model5 = build_model()  # nowy model
model5.fit(X_train_smotet, tf.keras.utils.to_categorical(y_train_smotet, num_classes=3), epochs=5, batch_size=32, verbose=1)  # wyszkol model
m5_eval = model5.evaluate(X_test, y_test, return_dict=True, verbose=0)  # zapisz metryki
print(classification_report(y_test.argmax(1), model5.predict(X_test, verbose=0).argmax(1), zero_division=0))  # raport


Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/normalization/batch_normalization.py:142: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


262/262 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 1.0611
Epoch 2/5
262/262 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.7334
Epoch 3/5
262/262 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.5852
Epoch 4/5
262/262 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5088
Epoch 5/5
262/262 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4626
              precision    recall  f1-score   support

           0       0.93      0.84      0.88       694
           1       0.78      0.73      0.75       215
           2       0.44      0.82      0.57        91

    accuracy                           0.81      1000
   macro avg       0.71      0.80      0.73      1000
weighted avg       0.85      0.81      0.82      1000



In [127]:
m5_eval

{'loss': 0.48849841952323914}

In [ ]:
assert 0.7 <= m5_eval['accuracy']
assert 0.65 <= m5_eval['f1_score']

In [128]:
print(classification_report(y_test.argmax(1), model5.predict(X_test, verbose=0).argmax(1), zero_division=0))  # raport


              precision    recall  f1-score   support

           0       0.93      0.84      0.88       694
           1       0.78      0.73      0.75       215
           2       0.44      0.82      0.57        91

    accuracy                           0.81      1000
   macro avg       0.71      0.80      0.73      1000
weighted avg       0.85      0.81      0.82      1000



<div class='alert alert-block alert-info'>
 W zależności od przebiegu uczenia, wyniki będą zbliżone lub nieznacznie odbiegające od tych, któe daje SMOTE. Znacząco powinna wzrosnąć czułość (wykrywalność) klas mniejszościowych w stosunku do scenariusza bazowego, kosztem precyzji. Innymi słowy - model częściej znajduje obiekty klasy mniejszościowej, ale jednocześnie zaczyna się mylić robiąć takie przypisania.
</div>

# Nadawanie wag klasom

Jeszcze jednym sposobem na szkolenie sieci neuronowej do rozpoznawania obiektów klasy mniejszościowej, jest nadanie wag poszczególnym klasom. Działa to w sposób następujący:

1. Każdej klasie nadajemy jakąś wagę.
2. Podczas procesu uczenia się, dla obiektów danej klasy, funkcja kosztu (np. entropia krzyżowa, ang. *Cross entropy*) jest wymnażana przez tą wagę
3. W ten sposób, obiekty określonej klasy mogą ważyć więcej lub mniej w przypadku ich błędnej klasyfikacji i tym samym silniej lub słabiej wpływać na dopasowanie funkcji kosztu.


W naszym przykładzie spróbujemy **bez równoważenia zbioru** nadać klasie o najmniejszej liczności (2) wagę = 0.5, drugiej mniej licznej klasie (1) wagę 0.35 i klasie więszkościowej wagę 0.15, aby położyć większy nacisk na 1 i 2.

<div class='alert alert-block alert-warning'>
    <b>Zadanie:</b> utwórz słównik, określający wagi poszczgólnych klas w sposób następujący:
    <il>
        <li>Klasa 0: waga 0.15</li>
        <li>Klasa 1: waga 0.35</li>
        <li>Klasa 2: waga 0.5</li>
    </il>
</div>

In [129]:
class_weights = {0: 0.15, 1: 0.35, 2: 0.5}  # wagi klas


<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Wykorzystując napisaną wcześniej funkcję- utwórz nowy obiekt sieci neurnowej</li>
        <li>Wyszkol przygotowany model przez 5 epok (batch size 32) na <b>PODSTAWOWYM zbiorze treningowym</b> bez równoważenia </li>
        <li>Do funkcji <code>fit()</code> sieci neuronowej, przekaż dodatkowy argument <code>class_weights=</code>zawierający określone wcześniej wagi klas.<li>
        <li>Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m6_eval</li>
        <li>Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.</li>
    </ol>
</div>

In [130]:
model6 = build_model()  # nowy model
model6.fit(X_train, y_train, epochs=5, batch_size=32, verbose=1, class_weight=class_weights)  # wyszkol z wagami
m6_eval = model6.evaluate(X_test, y_test, return_dict=True, verbose=0)  # zapisz metryki
print(classification_report(y_test.argmax(1), model6.predict(X_test, verbose=0).argmax(1), zero_division=0))  # raport


Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/normalization/batch_normalization.py:142: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.2727
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2204
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1957
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1832
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1624
              precision    recall  f1-score   support

           0       0.86      0.84      0.85       694
           1       0.68      0.52      0.59       215
           2       0.25      0.43      0.32        91

    accuracy                           0.74      1000
   macro avg       0.60      0.60      0.59      1000
weighted avg       0.77      0.74      0.75      1000



In [131]:
m6_eval

{'loss': 0.6721091866493225}

Sprawdzenie poprawności wyniku

In [133]:
print(classification_report(y_test.argmax(1), model6.predict(X_test, verbose=0).argmax(1), zero_division=0))  # raport

              precision    recall  f1-score   support

           0       0.86      0.84      0.85       694
           1       0.68      0.52      0.59       215
           2       0.25      0.43      0.32        91

    accuracy                           0.74      1000
   macro avg       0.60      0.60      0.59      1000
weighted avg       0.77      0.74      0.75      1000



<div class='alert alert-block alert-info'>
 Otrzymane wyniki nie wyglądają na istotnie lepsze/gorsze od tych, otrzymywanych podczas równoważenia zbioru. Zastosowanie wag dla klas jest po prostu kolejnym narzędziem, po które warto sięgnąć w sytuacji, gdy mamy do czynienia z niezbalansowanymi klasami w zbiorze uczącym.
</div>

# Dalsze eksperymenty

Jeśli chcesz, przepowadź dalesze eksperymenty na przedstawionym zbiorze danych, obejmujące np. poszukiwanie odpowiedniej architektury sieci oraz hiperparametrów. Spróbuj zastosować różne metody równoważenia zbiorów, z odmiennymi parametrami. Może uda Ci się uzyskać zadowalajace rezultaty?